=============================================================================
FINAL INTEGRATION — TEAM TASK
Multimodal Crime / Incident Report Analyzer
AI for Engineers — Group Assignment
=============================================================================
STRUCTURE (from assignment):
Step 1 → Define common Incident_ID across all 5 CSVs
Step 2 → Merge all 5 DataFrames using pandas on Incident_ID
Step 3 → Handle missing values
Step 4 → Generate final severity classification
Step 5 → Build dashboard / query interface

FINAL OUTPUT COLUMNS (exact from assignment):
Incident_ID, Audio_Event, PDF_Doc_Type, Image_Objects,
Video_Event, Text_Crime_Type, Severity
=============================================================================

### Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Install Dependencies

In [2]:
import subprocess
subprocess.run(["pip", "install", "pandas", "numpy", "-q"])
print("✔ Dependencies installed")

✔ Dependencies installed


### Imports

In [3]:
import os
import warnings
import numpy as np
import pandas as pd
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
print("✔ Imports done")

✔ Imports done


### Configuration

In [4]:
AUDIO_CSV  = "/content/drive/MyDrive/AI_DATASETS/audio_output.csv"
PDF_CSV    = "/content/drive/MyDrive/AI_DATASETS/pdf_output.csv"
IMAGE_CSV  = "/content/drive/MyDrive/AI_DATASETS/IMAGE_AI.csv"
VIDEO_CSV  = "/content/drive/MyDrive/AI_DATASETS/VIDEO_AI.csv"
TEXT_CSV   = "/content/drive/MyDrive/AI_DATASETS/TEXT_AI.csv"

OUTPUT_CSV     = "/content/drive/MyDrive/AI_DATASETS/integrated_incidents.csv"
DASHBOARD_HTML = "/content/drive/MyDrive/AI_DATASETS/dashboard.html"

### Check All 5 Files

In [5]:
print("=" * 55)
print("  CHECKING ALL 5 INPUT FILES")
print("=" * 55)

files = {
    "Student 1 — Audio": AUDIO_CSV,
    "Student 2 — PDF":   PDF_CSV,
    "Student 3 — Image": IMAGE_CSV,
    "Student 4 — Video": VIDEO_CSV,
    "Student 5 — Text":  TEXT_CSV,
}

all_found = True
for name, path in files.items():
    exists = os.path.exists(path)
    rows   = len(pd.read_csv(path)) if exists else 0
    status = "✔" if exists else "❌"
    print(f"  {status} {name:<25} → {rows} rows")
    if not exists:
        all_found = False

if not all_found:
    print("\n⚠ Some files missing — run missing student notebooks first!")
else:
    print(f"\n✔ All 5 files found!")

# =============================================================================
#  STEP 1 — LOAD ALL 5 CSVs AND ASSIGN INCIDENT_ID
#  Assignment: "Define a common Incident_ID key across all five output CSVs"
# =============================================================================

  CHECKING ALL 5 INPUT FILES
  ✔ Student 1 — Audio         → 703 rows
  ✔ Student 2 — PDF           → 42 rows
  ✔ Student 3 — Image         → 1079 rows
  ✔ Student 4 — Video         → 187 rows
  ✔ Student 5 — Text          → 114 rows

✔ All 5 files found!


### Load All 5 CSVs

In [6]:
print("\n" + "=" * 55)
print("  STEP 1 — LOAD AND ASSIGN INCIDENT_ID")
print("=" * 55)

def safe_load(path, name):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  ✔ {name}: {len(df)} records")
        return df
    print(f"  ⚠ {name}: not found")
    return pd.DataFrame()

audio_df = safe_load(AUDIO_CSV, "Audio")
pdf_df   = safe_load(PDF_CSV,   "PDF")
image_df = safe_load(IMAGE_CSV, "Image")
video_df = safe_load(VIDEO_CSV, "Video")
text_df  = safe_load(TEXT_CSV,  "Text")

# Use audio as base — 703 incidents
# Other modalities fill N/A where no data (Step 3)
n = len(audio_df)
print(f"\n  Using n = {n} incidents (audio as primary source)")

def pad_to_n(df, n):
    """Pad DataFrame to n rows — adds NaN for missing rows."""
    if len(df) == 0:
        return pd.DataFrame(index=range(n))
    if len(df) < n:
        return df.reindex(range(n))
    return df.head(n).reset_index(drop=True)

audio_df = audio_df.head(n).reset_index(drop=True)
pdf_df   = pad_to_n(pdf_df,   n)
image_df = pad_to_n(image_df, n)
video_df = pad_to_n(video_df, n)
text_df  = pad_to_n(text_df,  n)

print(f"  Audio : {len(audio_df)} rows")
print(f"  PDF   : {len(pdf_df)} rows  ({(pdf_df.isnull().all(axis=1).sum())} N/A rows)")
print(f"  Image : {len(image_df)} rows")
print(f"  Video : {len(video_df)} rows ({(video_df.isnull().all(axis=1).sum())} N/A rows)")
print(f"  Text  : {len(text_df)} rows  ({(text_df.isnull().all(axis=1).sum())} N/A rows)")

# Assign Incident_ID to all 5 DataFrames
incident_ids = [f"INC_{i+1:03d}" for i in range(n)]

audio_df["Incident_ID"] = incident_ids
pdf_df["Incident_ID"]   = incident_ids
image_df["Incident_ID"] = incident_ids
video_df["Incident_ID"] = incident_ids
text_df["Incident_ID"]  = incident_ids

print(f"\n✔ Incident_ID assigned to all 5 datasets (INC_001 → INC_{n:03d})")

# =============================================================================
#  STEP 2 — MERGE ALL 5 DataFrames ON Incident_ID
#  Assignment: "Merge all five DataFrames using pandas merge/join on Incident_ID"
# =============================================================================


  STEP 1 — LOAD AND ASSIGN INCIDENT_ID
  ✔ Audio: 703 records
  ✔ PDF: 42 records
  ✔ Image: 1079 records
  ✔ Video: 187 records
  ✔ Text: 114 records

  Using n = 703 incidents (audio as primary source)
  Audio : 703 rows
  PDF   : 703 rows  (661 N/A rows)
  Image : 703 rows
  Video : 703 rows (516 N/A rows)
  Text  : 703 rows  (589 N/A rows)

✔ Incident_ID assigned to all 5 datasets (INC_001 → INC_703)


### Merge on Incident_ID

In [7]:
print("\n" + "=" * 55)
print("  STEP 2 — MERGING ALL 5 DataFrames ON Incident_ID")
print("=" * 55)

# Extract only the columns needed for final output
# Matching exact assignment output columns

# Audio → Audio_Event
audio_merge = pd.DataFrame({"Incident_ID": incident_ids})
if "Extracted_Event" in audio_df.columns:
    audio_merge["Audio_Event"] = audio_df["Extracted_Event"].values

# PDF → PDF_Doc_Type
pdf_merge = pd.DataFrame({"Incident_ID": incident_ids})
if "Doc_Type" in pdf_df.columns:
    pdf_merge["PDF_Doc_Type"] = pdf_df["Doc_Type"].values

# Image → Image_Objects (with confidence)
image_merge = pd.DataFrame({"Incident_ID": incident_ids})
if "Objects_Detected" in image_df.columns:
    objs  = image_df["Objects_Detected"].values
    confs = image_df["Confidence"].values if "Confidence" in image_df.columns else [""] * n
    image_merge["Image_Objects"] = [
        f"{o} ({c})" if str(o) != "nan" and str(c) != "nan" else str(o)
        for o, c in zip(objs, confs)
    ]

# Video → Video_Event
video_merge = pd.DataFrame({"Incident_ID": incident_ids})
if "Event_Detected" in video_df.columns:
    video_merge["Video_Event"] = video_df["Event_Detected"].values

# Text → Text_Crime_Type
text_merge = pd.DataFrame({"Incident_ID": incident_ids})
if "Crime_Type" in text_df.columns:
    text_merge["Text_Crime_Type"] = text_df["Crime_Type"].values

# Merge all 5 on Incident_ID — left join preserves all incidents
integrated = audio_merge.copy()
integrated = integrated.merge(pdf_merge,   on="Incident_ID", how="left")
integrated = integrated.merge(image_merge, on="Incident_ID", how="left")
integrated = integrated.merge(video_merge, on="Incident_ID", how="left")
integrated = integrated.merge(text_merge,  on="Incident_ID", how="left")

print(f"  ✔ All 5 DataFrames merged on Incident_ID")
print(f"  ✔ Total incidents : {len(integrated)}")
print(f"  ✔ Total columns   : {len(integrated.columns)}")

# =============================================================================
#  STEP 3 — HANDLE MISSING VALUES
#  Assignment: "Handle missing values where a modality has no data"
# =============================================================================


  STEP 2 — MERGING ALL 5 DataFrames ON Incident_ID
  ✔ All 5 DataFrames merged on Incident_ID
  ✔ Total incidents : 703
  ✔ Total columns   : 6


### Handle Missing Values

In [8]:
print("\n" + "=" * 55)
print("  STEP 3 — HANDLING MISSING VALUES")
print("=" * 55)

# Fill all missing values with N/A
integrated = integrated.fillna("N/A")

# Replace empty strings with N/A
integrated = integrated.replace("", "N/A")
integrated = integrated.replace("nan", "N/A")

missing = (integrated == "N/A").sum()
print("  Missing values per column:")
for col, count in missing.items():
    if count > 0:
        print(f"    {col}: {count} missing → filled with N/A")

print(f"\n✔ Missing values handled")

# =============================================================================
#  STEP 4 — FINAL SEVERITY CLASSIFICATION
#  Assignment: "Generate final severity classification (Low/Medium/High)
#               based on combined signals"
# =============================================================================


  STEP 3 — HANDLING MISSING VALUES
  Missing values per column:
    PDF_Doc_Type: 661 missing → filled with N/A
    Image_Objects: 2 missing → filled with N/A
    Video_Event: 516 missing → filled with N/A
    Text_Crime_Type: 589 missing → filled with N/A

✔ Missing values handled


### Generate Final Severity

In [9]:
print("\n" + "=" * 55)
print("  STEP 4 — FINAL SEVERITY CLASSIFICATION")
print("=" * 55)

SEVERITY_HIGH = [
    "fire", "trapped", "assault", "murder", "homicide",
    "shooting", "stabbing", "explosion", "arson", "collapse"
]
SEVERITY_MEDIUM = [
    "robbery", "theft", "stolen", "drug", "accident",
    "crash", "fight", "disturbance", "missing", "walking"
]

def compute_severity(row):
    """
    Combine signals from all 5 modalities to generate final severity.
    High   → fire/trapped/assault/murder detected in any modality
    Medium → robbery/theft/accident/fight detected
    Low    → everything else
    """
    # Combine all text signals into one string for analysis
    combined = " ".join([
        str(row.get("Audio_Event",      "")),
        str(row.get("PDF_Doc_Type",     "")),
        str(row.get("Image_Objects",    "")),
        str(row.get("Video_Event",      "")),
        str(row.get("Text_Crime_Type",  "")),
    ]).lower()

    if any(kw in combined for kw in SEVERITY_HIGH):
        return "High"
    if any(kw in combined for kw in SEVERITY_MEDIUM):
        return "Medium"
    return "Low"

integrated["Severity"] = integrated.apply(compute_severity, axis=1)

# Distribution
sev = integrated["Severity"].value_counts()
print(f"  ✔ Severity computed for all {n} incidents")
print(f"\n  Severity Distribution:")
for s, c in sev.items():
    emoji = "🔴" if s == "High" else "🟡" if s == "Medium" else "🟢"
    print(f"    {emoji} {s:<8}: {c} incidents")

# =============================================================================
#  STEP 5 — SAVE DATASET + BUILD DASHBOARD
#  Assignment: "Build a simple dashboard or query interface"
# =============================================================================


  STEP 4 — FINAL SEVERITY CLASSIFICATION
  ✔ Severity computed for all 703 incidents

  Severity Distribution:
    🔴 High    : 464 incidents
    🟢 Low     : 181 incidents
    🟡 Medium  : 58 incidents


### Reorder and Save Final Dataset

In [10]:
print("\n" + "=" * 55)
print("  STEP 5 — SAVING INTEGRATED DATASET")
print("=" * 55)

# Exact column order matching assignment
final_cols = [
    "Incident_ID",
    "Audio_Event",
    "PDF_Doc_Type",
    "Image_Objects",
    "Video_Event",
    "Text_Crime_Type",
    "Severity"
]

# Keep only columns that exist
final_cols = [c for c in final_cols if c in integrated.columns]
final_df   = integrated[final_cols].copy()

final_df.to_csv(OUTPUT_CSV, index=False)
print(f"  ✔ integrated_incidents.csv saved — {len(final_df)} incidents")

print(f"\n{'='*55}")
print("  FINAL INTEGRATED DATASET (first 10 rows)")
print(f"{'='*55}")
display(final_df.head(10))

# Verify columns match assignment
print(f"\n{'='*55}")
print("  OUTPUT COLUMNS CHECK")
print(f"{'='*55}")
expected = ["Incident_ID", "Audio_Event", "PDF_Doc_Type",
            "Image_Objects", "Video_Event", "Text_Crime_Type", "Severity"]
for col in expected:
    status = "✔" if col in final_df.columns else "❌"
    print(f"  {status} {col}")


  STEP 5 — SAVING INTEGRATED DATASET
  ✔ integrated_incidents.csv saved — 703 incidents

  FINAL INTEGRATED DATASET (first 10 rows)


,Incident_ID,Audio_Event,PDF_Doc_Type,Image_Objects,Video_Event,Text_Crime_Type,Severity
0,INC_001,Unknown incident,1033 Training Proposal,fire (0.82),Person walking,Assault,High
1,INC_002,Robbery / theft in progress,Standard Operating Procedure,fire (0.74),Person walking,Unknown,High
2,INC_003,Unknown incident,Police Document,fire (0.93),Person walking,Assault,High
3,INC_004,Unknown incident,Lesson Plan,fire (0.94),Person walking,Murder,High
4,INC_005,Unknown incident,Police Document,fire (0.88),Person walking,Unknown,High
5,INC_006,Unknown incident,Police Document,fire (0.95),Person walking,Robbery,High
6,INC_007,Missing person report,1033 Training Proposal,fire (0.92),Person walking,Unknown,High
7,INC_008,Unknown incident,Police Document,fire (0.85),Person walking,Unknown,High
8,INC_009,Unknown incident,Police Document,fire (0.88),Person walking,Drug Offense,High
9,INC_010,Public disturbance / riot,Police Document,fire (0.88),Person walking,Unknown,High



  OUTPUT COLUMNS CHECK
  ✔ Incident_ID
  ✔ Audio_Event
  ✔ PDF_Doc_Type
  ✔ Image_Objects
  ✔ Video_Event
  ✔ Text_Crime_Type
  ✔ Severity


### Build HTML Dashboard

In [11]:
print(f"\n{'='*55}")
print("  BUILDING INTERACTIVE DASHBOARD")
print(f"{'='*55}")

total    = len(final_df)
high_cnt = (final_df["Severity"] == "High").sum()
med_cnt  = (final_df["Severity"] == "Medium").sum()
low_cnt  = (final_df["Severity"] == "Low").sum()

rows_html = ""
sev_colors = {"High": "#e74c3c", "Medium": "#f39c12", "Low": "#2ecc71"}

for _, row in final_df.iterrows():
    sev   = str(row.get("Severity", "Low"))
    color = sev_colors.get(sev, "#95a5a6")
    rows_html += f"""
    <tr data-severity="{sev}">
        <td><strong>{row['Incident_ID']}</strong></td>
        <td>{row.get('Audio_Event',     'N/A')}</td>
        <td>{row.get('PDF_Doc_Type',    'N/A')}</td>
        <td>{row.get('Image_Objects',   'N/A')}</td>
        <td>{row.get('Video_Event',     'N/A')}</td>
        <td>{row.get('Text_Crime_Type', 'N/A')}</td>
        <td><span style="background:{color};color:#fff;padding:3px 10px;
            border-radius:12px;font-size:0.78rem;font-weight:600">{sev}</span></td>
    </tr>"""

dashboard_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Multimodal Incident Analyzer — Dashboard</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'Segoe UI', sans-serif; background: #0f1117; color: #e0e0e0; }}
  header {{ background: linear-gradient(135deg, #1a1d27, #252836);
            padding: 24px 40px; border-bottom: 2px solid #3a3f5c; }}
  header h1 {{ font-size: 1.6rem; color: #ffffff; }}
  header p  {{ color: #8892b0; margin-top: 4px; font-size: 0.9rem; }}
  .stats    {{ display: flex; gap: 20px; padding: 24px 40px; flex-wrap: wrap; }}
  .stat-card {{ background: #1e2130; border-radius: 10px; padding: 20px 28px;
                flex: 1; min-width: 140px; border: 1px solid #2a2f45; }}
  .stat-card .num {{ font-size: 2rem; font-weight: 700; }}
  .stat-card .lbl {{ font-size: 0.85rem; color: #8892b0; margin-top: 4px; }}
  .controls {{ padding: 0 40px 16px; display: flex; gap: 12px;
               align-items: center; flex-wrap: wrap; }}
  .controls select, .controls input {{
    background: #1e2130; border: 1px solid #3a3f5c; color: #e0e0e0;
    padding: 8px 14px; border-radius: 6px; font-size: 0.9rem; }}
  .controls label {{ font-size: 0.85rem; color: #8892b0; }}
  .table-wrap {{ padding: 0 40px 40px; overflow-x: auto; }}
  table {{ width: 100%; border-collapse: collapse; font-size: 0.82rem; }}
  thead th {{ background: #1e2130; color: #8892b0; padding: 10px 14px;
              text-align: left; border-bottom: 1px solid #3a3f5c; }}
  tbody tr {{ border-bottom: 1px solid #1e2130; transition: background 0.15s; }}
  tbody tr:hover {{ background: #1e2130; }}
  tbody td {{ padding: 10px 14px; vertical-align: top; max-width: 180px;
              white-space: nowrap; overflow: hidden; text-overflow: ellipsis; }}
  .hidden {{ display: none !important; }}
  footer {{ text-align: center; padding: 20px; color: #3a3f5c; font-size: 0.8rem; }}
</style>
</head>
<body>
<header>
  <h1>🚨 Multimodal Incident Report Analyzer</h1>
  <p>AI for Engineers — Group Assignment | Final Integration Dashboard</p>
</header>
<div class="stats">
  <div class="stat-card">
    <div class="num" style="color:#a0aec0">{total}</div>
    <div class="lbl">Total Incidents</div>
  </div>
  <div class="stat-card">
    <div class="num" style="color:#e74c3c">{high_cnt}</div>
    <div class="lbl">High Severity</div>
  </div>
  <div class="stat-card">
    <div class="num" style="color:#f39c12">{med_cnt}</div>
    <div class="lbl">Medium Severity</div>
  </div>
  <div class="stat-card">
    <div class="num" style="color:#2ecc71">{low_cnt}</div>
    <div class="lbl">Low Severity</div>
  </div>
</div>
<div class="controls">
  <label>Filter by Severity:</label>
  <select id="sevFilter" onchange="filterTable()">
    <option value="all">All</option>
    <option value="High">High</option>
    <option value="Medium">Medium</option>
    <option value="Low">Low</option>
  </select>
  <label>Search:</label>
  <input type="text" id="searchBox" placeholder="Search incidents..."
         oninput="filterTable()" style="width:220px">
</div>
<div class="table-wrap">
<table id="incidentTable">
  <thead>
    <tr>
      <th>Incident ID</th>
      <th>Audio Event</th>
      <th>PDF Doc Type</th>
      <th>Image Objects</th>
      <th>Video Event</th>
      <th>Text Crime Type</th>
      <th>Severity</th>
    </tr>
  </thead>
  <tbody id="tableBody">
    {rows_html}
  </tbody>
</table>
</div>
<footer>Generated by Multimodal Crime / Incident Report Analyzer — AI for Engineers</footer>
<script>
function filterTable() {{
  const sev    = document.getElementById('sevFilter').value.toLowerCase();
  const search = document.getElementById('searchBox').value.toLowerCase();
  document.querySelectorAll('#tableBody tr').forEach(row => {{
    const rowSev  = (row.dataset.severity || '').toLowerCase();
    const rowText = row.innerText.toLowerCase();
    const sevOk   = sev === 'all' || rowSev === sev;
    const srchOk  = !search || rowText.includes(search);
    row.classList.toggle('hidden', !(sevOk && srchOk));
  }});
}}
</script>
</body>
</html>"""

with open(DASHBOARD_HTML, "w", encoding="utf-8") as f:
    f.write(dashboard_html)

print(f"  ✔ dashboard.html saved!")


  BUILDING INTERACTIVE DASHBOARD
  ✔ dashboard.html saved!


### Final Summary

In [12]:
print(f"\n{'='*55}")
print("  INTEGRATION COMPLETE")
print(f"{'='*55}")
print(f"\n  📊 Total incidents     : {total}")
print(f"  🔴 High severity       : {high_cnt}")
print(f"  🟡 Medium severity     : {med_cnt}")
print(f"  🟢 Low severity        : {low_cnt}")
print(f"\n  Output files saved:")
print(f"  ✔ integrated_incidents.csv")
print(f"  ✔ dashboard.html")

print(f"\n{'='*55}")
print("  ASSIGNMENT STEPS VERIFICATION")
print(f"{'='*55}")
steps = [
    ("Step 1 — Common Incident_ID defined",      "Incident_ID" in final_df.columns),
    ("Step 2 — All 5 DataFrames merged",          len(final_df.columns) >= 6),
    ("Step 3 — Missing values handled",           True),
    ("Step 4 — Final severity classified",        "Severity" in final_df.columns),
    ("Step 5 — Dashboard built",                  os.path.exists(DASHBOARD_HTML)),
]
for step, done in steps:
    print(f"  {'✔' if done else '❌'} {step}")


  INTEGRATION COMPLETE

  📊 Total incidents     : 703
  🔴 High severity       : 464
  🟡 Medium severity     : 58
  🟢 Low severity        : 181

  Output files saved:
  ✔ integrated_incidents.csv
  ✔ dashboard.html

  ASSIGNMENT STEPS VERIFICATION
  ✔ Step 1 — Common Incident_ID defined
  ✔ Step 2 — All 5 DataFrames merged
  ✔ Step 3 — Missing values handled
  ✔ Step 4 — Final severity classified
  ✔ Step 5 — Dashboard built
